In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import plot_tree

### Load the data

In [ ]:
data = pd.read_csv('mushrooms.csv')
data.head()

### List unique values in each column

In [ ]:
labels = pd.DataFrame(index=data.columns)
labels['values'] = [','.join(data[c].unique()) for c in data.columns]
labels['numvalues'] = [len(v.split(',')) for v in labels['values']]
labels

### Inputs and output

In [ ]:
X = data.drop(columns='class')
y = data['class']

### Encode the input

In [ ]:
encoders = dict()
encXdict = dict()

for column in X.columns:
    num_unique = labels.loc[column,'numvalues']

    if num_unique==2:
        myencoder = LabelBinarizer()
        encoders[column] = myencoder
        encXdict[column] = myencoder.fit_transform(X[[column]])
        encXdict[column] = encXdict[column][:,0]
    elif num_unique>2:
        myencoder = OneHotEncoder(sparse_output=False)
        encoders[column] = myencoder
        e = myencoder.fit_transform(X[[column]])
        for i, col in enumerate(myencoder.get_feature_names_out([column])):
            encXdict[col] = e[:,i]
    else:
        print(f"WARNING! Ignoring {column} because it has only {num_unique} unique value.")

Xdf = pd.DataFrame(encXdict)

### Split off the test data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Xdf, y, test_size = 0.3, random_state = 345)

### Build a decision tree

In [ ]:
model = DecisionTreeClassifier(criterion='gini',max_depth=10).fit(X_train, y_train)

### Plot it

In [ ]:
plt.figure(figsize=(30,18))
plot_tree(model,fontsize=16, feature_names=list(Xdf.columns),filled=True);

### Test accuracy

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:',accuracy_score(y_test, y_pred))

### Feature importance

In [ ]:
fimp = np.argsort(model.feature_importances_)
fimp = fimp[::-1]
sorted_features = Xdf.columns[fimp]
sorted_feature_importance = model.feature_importances_[fimp]

fig = plt.figure(figsize=(12,2))
ax = plt.subplot()
ax.stem(sorted_features,sorted_feature_importance)
ax.set_xlim(-.1,10)
ax.set_ylim(0,0.7)
ax.tick_params(axis='x',rotation=45)
ax.spines[['top','right']].set_visible(False)